# Phase 1B — Daily Event-Window Label Construction (Option A)
## Matching Sovereign Bond Yield Changes to Central Bank Speech Dates

**What this notebook does:**
1. Loads daily sovereign bond yields from `sovereign_bond_yields.csv`
2. Computes daily basis-point changes (Δy₂)
3. Matches labels to documents and chunks from Phase 1
4. Produces labeled datasets for Phase 2 & 3

**Label Definition (Option A — daily frequency):**
- Δy₂ = US_2Y(t) - US_2Y(t-1) in basis points
- where t = the date of the speech/event

---

In [ ]:
import sys
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
print(f'Project root: {PROJECT_ROOT}')

---
## Step 1: Load Sovereign Bond Yield Data

In [ ]:
# Load the sovereign bond yields CSV (produced by Ticker_Fetch.ipynb)
yields = pd.read_csv(PROJECT_ROOT / 'notebooks' / 'sovereign_bond_yields.csv', parse_dates=['date'])
yields = yields.sort_values('date').reset_index(drop=True)

print(f'Yield data: {len(yields)} rows')
print(f'Date range: {yields["date"].min().date()} to {yields["date"].max().date()}')
print(f'Columns: {list(yields.columns)}')
yields.tail()

---
## Step 2: Compute Daily Yield Changes (Basis Points)

In [ ]:
# Compute daily changes in basis points
df = yields.set_index('date').sort_index()

yield_changes = pd.DataFrame(index=df.index)

# Primary label: US 2Y daily change (basis points)
# Note: yields are in percentage points, so diff * 100 = basis points
yield_changes['delta_US_2Y_bp'] = df['US_2Y'].diff() * 100

# Secondary labels
yield_changes['delta_US_10Y_bp'] = df['US_10Y'].diff() * 100

# 2s10s slope
slope = df['US_10Y'] - df['US_2Y']
yield_changes['delta_slope_2s10s_bp'] = slope.diff() * 100

# Euro Area
if 'EA_2Y' in df.columns:
    yield_changes['delta_EA_2Y_bp'] = df['EA_2Y'].diff() * 100
if 'EA_10Y' in df.columns:
    yield_changes['delta_EA_10Y_bp'] = df['EA_10Y'].diff() * 100

# Germany, UK, Japan
if 'DE_10Y' in df.columns:
    yield_changes['delta_DE_10Y_bp'] = df['DE_10Y'].diff() * 100
if 'GB_10Y' in df.columns:
    yield_changes['delta_GB_10Y_bp'] = df['GB_10Y'].diff() * 100
if 'JP_10Y' in df.columns:
    yield_changes['delta_JP_10Y_bp'] = df['JP_10Y'].diff() * 100

print(f'Yield changes computed: {len(yield_changes)} days')
print(f'Columns: {list(yield_changes.columns)}')
print(f'\nUS 2Y daily change stats:')
print(yield_changes['delta_US_2Y_bp'].describe())

In [ ]:
# Visualize the distribution of daily yield changes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(yield_changes['delta_US_2Y_bp'].dropna(), bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Daily Change (bp)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Daily US 2Y Yield Changes')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)

# Time series
axes[1].plot(yield_changes.index, yield_changes['delta_US_2Y_bp'], linewidth=0.3, alpha=0.7)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Daily Change (bp)')
axes[1].set_title('US 2Y Yield Daily Changes Over Time')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'processed' / 'yield_changes_distribution.png'), dpi=150)
plt.show()

---
## Step 3: Load Document Registry from Phase 1

In [ ]:
# Load the document registry produced by Phase 1
registry = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'document_registry.parquet')
registry['date'] = pd.to_datetime(registry['date'])

print(f'Document registry: {len(registry)} documents')
print(f'Date range: {registry["date"].min().date()} to {registry["date"].max().date()}')
print(f'Central banks: {registry["central_bank"].nunique()}')
print(f'\nTop 10 central banks:')
print(registry['central_bank'].value_counts().head(10))

---
## Step 4: Match Yield Change Labels to Documents

In [ ]:
# For each speech date, assign the yield change on that day
# If speech falls on a non-trading day, use the next available trading day

trading_dates = yield_changes.index.sort_values()

def find_yield_change(doc_date):
    """Find the yield change for a given date (or next trading day)."""
    mask = trading_dates >= doc_date
    if mask.any():
        match_date = trading_dates[mask][0]
        return yield_changes.loc[match_date]
    return pd.Series({col: np.nan for col in yield_changes.columns})

print('Matching labels to documents (this may take a minute)...')
labels_matched = registry['date'].apply(find_yield_change)
labeled_docs = pd.concat([registry, labels_matched], axis=1)

# Stats
primary = 'delta_US_2Y_bp'
valid = labeled_docs[primary].notna().sum()
total = len(labeled_docs)
print(f'\nLabel matching complete:')
print(f'  Documents with valid Δy₂: {valid}/{total} ({valid/total*100:.1f}%)')
print(f'  Label distribution:')
print(labeled_docs[primary].describe())

---
## Step 5: Propagate Labels to Chunk Registry

In [ ]:
# Load chunk registry
chunk_path = PROJECT_ROOT / 'data' / 'processed' / 'chunk_registry_paragraph.parquet'
chunks = pd.read_parquet(chunk_path)
print(f'Chunks loaded: {len(chunks)}')

# Label columns to propagate
label_columns = [c for c in yield_changes.columns if c in labeled_docs.columns]
doc_labels = labeled_docs[['doc_id'] + label_columns].copy()

# Merge onto chunks
labeled_chunks = chunks.merge(doc_labels, on='doc_id', how='left')

valid_chunks = labeled_chunks[primary].notna().sum()
print(f'Chunks with valid Δy₂: {valid_chunks}/{len(labeled_chunks)} ({valid_chunks/len(labeled_chunks)*100:.1f}%)')

---
## Step 6: Save Labeled Datasets

In [ ]:
# Save everything
output_dir = PROJECT_ROOT / 'data' / 'processed'

labeled_docs.to_parquet(output_dir / 'labeled_document_registry.parquet', index=False)
labeled_chunks.to_parquet(output_dir / 'labeled_chunk_registry.parquet', index=False)
yield_changes.to_parquet(output_dir / 'daily_yield_changes.parquet')

print('Saved:')
print(f'  ✓ labeled_document_registry.parquet ({len(labeled_docs)} docs)')
print(f'  ✓ labeled_chunk_registry.parquet ({len(labeled_chunks)} chunks)')
print(f'  ✓ daily_yield_changes.parquet ({len(yield_changes)} days)')
print(f'\n✅ Phase 1B Complete — Labels ready for Phase 2')

---
## Quality Audit: Fed-Only Label Distribution

In [ ]:
# Focus on Federal Reserve speeches — most relevant for US 2Y yield
fed_docs = labeled_docs[labeled_docs['central_bank'] == 'Board of Governors of the Federal Reserve']
fed_labeled = fed_docs[fed_docs[primary].notna()]

print(f'Federal Reserve speeches with Δy₂ labels: {len(fed_labeled)}')
print(f'\nLabel stats (Fed-only):')
print(fed_labeled[primary].describe())

# Verify: mean should be ~0 (no unconditional bias)
mean_bp = fed_labeled[primary].mean()
print(f'\n✓ Mean Δy₂ = {mean_bp:.3f} bp (should be ≈0, no bias)')

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(fed_labeled[primary], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Δy₂ (basis points)')
ax.set_ylabel('Frequency')
ax.set_title(f'Distribution of Daily US 2Y Yield Changes on Fed Speech Days (N={len(fed_labeled)})')
ax.axvline(0, color='red', linestyle='--', alpha=0.7, label='Zero')
ax.legend()
plt.tight_layout()
plt.show()